In [ ]:
from google.colab import drive
import os
import sys

#zamontowanie dysku
drive.mount('/content/drive')

#ścieżki
DRIVE_PATH = '/content/drive/MyDrive/BetaZone-data'
REPO_NAME = 'e2e-climbing-vision'  # Poprawna nazwa folderu repozytorium
REPO_URL = f'https://github.com/jeicam3/{REPO_NAME}'

#klonowanie/aktalizacja repo
if not os.path.exists(f'/content/{REPO_NAME}'):
    !git clone {REPO_URL}
else:
    !git -C /content/{REPO_NAME} pull

#dodanie repozytorium
if f'/content/{REPO_NAME}' not in sys.path:
    sys.path.append(f'/content/{REPO_NAME}')

#kopiowanie i rozpakowanie danych z google drive
!cp "{DRIVE_PATH}/full-data.zip" /content/full-data.zip
!cp "{DRIVE_PATH}/labels.csv" /content/labels.csv

!mkdir -p /content/full-data
!unzip -q -o /content/full-data.zip -d /content/full-data

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
import copy

#importy z repo
from models.dataset import ClimbingDataset, ApplyTransform
from models.efficientnet import get_climbing_model

#tranformacje
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

#loadery
img_dir = '/content/full-data/full-data'

full_dataset = ClimbingDataset(csv_file='/content/labels.csv', img_dir=img_dir)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_subset, val_subset = random_split(full_dataset, [train_size, val_size])

train_data = ApplyTransform(train_subset, transform=train_transforms)
val_data = ApplyTransform(val_subset, transform=val_transforms)

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
val_loader = DataLoader(val_data, batch_size=16, shuffle=False)

#inicjalizacja modelu
model = get_climbing_model()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Próbki: Treningowe {len(train_data)}, Walidacyjne: {len(val_data)}")

In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

num_epochs = 40
patience = 8
counter = 0
best_val_loss = float('inf')
best_model_wts = copy.deepcopy(model.state_dict())

print(f"Start treningu na: {device}\n" + "-"*30)

for epoch in range(num_epochs):
    #trening
    model.train()
    running_train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device).float()
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader)

    #walidacja
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device).float()
            outputs = model(images)
            v_loss = criterion(outputs, labels)
            running_val_loss += v_loss.item()

    avg_val_loss = running_val_loss / len(val_loader)

    scheduler.step(avg_val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch [{epoch+1:02d}/{num_epochs}] | LR: {current_lr:.6f} | Loss: T {avg_train_loss:.4f} / V {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        print(f"Nowy najlepszy wynik")
        best_val_loss = avg_val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), 'climbing_model_best.pth')
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print(f"\nEARLY STOPPING po {epoch+1} epokach.")
            break

model.load_state_dict(best_model_wts)
print("-" * 30 + f"\nKoniec. Najlepszy wynik: {best_val_loss:.4f}")

In [ ]:
MODEL_NAME = "climbing_model_best.pth"
CHECKPOINT_DIR = f"{DRIVE_PATH}/checkpoints"

# Utworzenie folderu na Drive jeśli nie istnieje
!mkdir -p {CHECKPOINT_DIR}

# Kopia na Google Drive
!cp {MODEL_NAME} {CHECKPOINT_DIR}/{MODEL_NAME}

print(f"Model zapisany w: {CHECKPOINT_DIR}/{MODEL_NAME}")